# Netflix Movies and TV Shows EDA

## Project Introduction

This notebook presents a professional Exploratory Data Analysis (EDA) of the Netflix Movies and TV Shows dataset from Kaggle. The analysis investigates Netflix's content catalog across content type, time, geography, genre, ratings, duration, directors, and cast.

The goal is to convert raw entertainment catalog data into clear business insights that can support content strategy, regional planning, audience targeting, and portfolio decisions.

## Business Objective

Netflix operates in a highly competitive streaming market where content breadth, regional relevance, genre mix, and audience suitability are central to user acquisition and retention.

This EDA answers the following business questions:

- What proportion of the catalog is Movies vs TV Shows?
- How has Netflix content grown over time?
- Which countries contribute the most content?
- Which genres dominate the platform?
- What ratings are most common across Movies and TV Shows?
- How long are movies and how many seasons do TV Shows usually have?
- Which directors and actors appear most frequently?
- What actionable recommendations can be made from these patterns?

## Data Loading

In [ ]:
# Import core data analysis and visualization libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

# Professional visualization defaults
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.dpi"] = 120

# Reusable project styling
BRAND_COLORS = {
    "red": "#E50914",
    "black": "#221F1F",
    "gray": "#6B7280",
    "blue": "#2F6B9A",
    "green": "#2CA58D",
    "orange": "#F58518",
}


def polish_chart(title, xlabel=None, ylabel=None, rotate_x=False):
    # Apply consistent labels, gridlines, and spacing to charts.
    plt.title(title, fontsize=16, fontweight="bold", pad=14)
    if xlabel is not None:
        plt.xlabel(xlabel)
    if ylabel is not None:
        plt.ylabel(ylabel)
    if rotate_x:
        plt.xticks(rotation=45, ha="right")
    sns.despine(left=False, bottom=False)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()


def add_bar_labels(ax, fmt="{:,.0f}"):
    # Add readable labels to vertical bar charts.
    for container in ax.containers:
        labels = [fmt.format(value) for value in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)


# Create output folder for saved charts
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)

In [ ]:
# Locate the Kaggle dataset from common project locations
current_dir = Path.cwd()
candidate_paths = [
    current_dir / "netflix_titles.csv",
    current_dir / "data" / "netflix_titles.csv",
    current_dir.parent / "netflix_titles.csv",
    current_dir.parent / "data" / "netflix_titles.csv",
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find netflix_titles.csv. Place it in the project root or in a data/ folder, then rerun this notebook."
    )

df = pd.read_csv(data_path)
print(f"Dataset loaded successfully from: {data_path}")

## Data Understanding

In [ ]:
# Display the first 5 rows
df.head()

In [ ]:
# Dataset shape and column names
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric and categorical columns
display(df.describe(include="all").T)

In [ ]:
# Missing values analysis
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_percentage=lambda x: (x["missing_count"] / len(df) * 100).round(2))
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=missing_summary.reset_index(),
    x="missing_percentage",
    y="index",
    color="#4C78A8",
)
plt.title("Missing Values by Column")
plt.xlabel("Missing Percentage")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(f'''
**Data Understanding Insights**

- The dataset contains **{df.shape[0]:,} titles** and **{df.shape[1]} columns**.
- Missing-value checks identify which fields need cleaning before analysis.
- Text-heavy columns such as director, cast, and country commonly require careful handling because missing values often mean unavailable metadata rather than true zero values.

**Business implication:** Metadata completeness affects search, recommendation quality, geographic analysis, and talent-level insights. Columns with high missingness should be interpreted carefully.
'''))

## Data Cleaning

In [ ]:
# Work on a copy to preserve the raw dataframe
netflix = df.copy()

# Standardize column names just in case whitespace exists
netflix.columns = netflix.columns.str.strip()

# Remove exact duplicate rows
duplicate_count = netflix.duplicated().sum()
netflix = netflix.drop_duplicates()
print(f"Duplicate rows removed: {duplicate_count}")

In [ ]:
# Fill missing categorical text fields with clear labels
fill_values = {
    "director": "Unknown Director",
    "cast": "Unknown Cast",
    "country": "Unknown Country",
    "rating": "Unknown Rating",
    "duration": "Unknown Duration",
}

netflix = netflix.fillna(fill_values)

# date_added is analytically important, so convert it to datetime
netflix["date_added"] = pd.to_datetime(netflix["date_added"], errors="coerce")

# Extract useful time features
netflix["year_added"] = netflix["date_added"].dt.year
netflix["month_added"] = netflix["date_added"].dt.month
netflix["month_name_added"] = netflix["date_added"].dt.month_name()

# Release year as numeric
netflix["release_year"] = pd.to_numeric(netflix["release_year"], errors="coerce")

In [ ]:
# Separate duration into value and unit
duration_parts = netflix["duration"].astype(str).str.extract(r"(\d+)\s*(\D+)")
netflix["duration_value"] = pd.to_numeric(duration_parts[0], errors="coerce")
netflix["duration_unit"] = duration_parts[1].str.strip()

# Create clean list versions for multi-value columns
netflix["country_clean"] = netflix["country"].apply(
    lambda value: [item.strip() for item in str(value).split(",") if item.strip()]
)
netflix["genre_clean"] = netflix["listed_in"].apply(
    lambda value: [item.strip() for item in str(value).split(",") if item.strip()]
)

# Create primary country and primary genre for simpler grouped analysis
netflix["primary_country"] = netflix["country_clean"].apply(lambda values: values[0] if values else "Unknown Country")
netflix["primary_genre"] = netflix["genre_clean"].apply(lambda values: values[0] if values else "Unknown Genre")

# Age of content when added to Netflix
netflix["content_age_when_added"] = netflix["year_added"] - netflix["release_year"]

netflix.head()

In [ ]:
# Verify cleaned dataset
cleaning_summary = pd.DataFrame({
    "rows_after_cleaning": [len(netflix)],
    "columns_after_cleaning": [netflix.shape[1]],
    "remaining_duplicates": [netflix.duplicated().sum()],
    "missing_date_added": [netflix["date_added"].isna().sum()],
})

display(cleaning_summary)
display(netflix[["type", "title", "date_added", "year_added", "month_added", "duration", "duration_value", "duration_unit", "primary_country", "primary_genre"]].head())

In [ ]:
display(Markdown(f'''
**Data Cleaning Insights**

- Removed **{duplicate_count:,} duplicate rows**.
- Converted `date_added` into a datetime field and extracted `year_added` and `month_added`.
- Split `duration` into numeric and categorical components so Movies and TV Shows can be analyzed separately.
- Created clean list-based country and genre columns to support multi-label analysis.

**Business implication:** Clean time, geography, genre, and duration fields make the catalog easier to analyze for acquisition trends, market coverage, and audience segmentation.
'''))

## Feature Engineering

Strong EDA is more useful when raw columns are converted into business-ready features. This section creates additional variables for catalog age, release era, audience maturity, international availability, and metadata quality.

In [ ]:
# Feature engineering for advanced analysis
latest_release_year = int(netflix["release_year"].max())

netflix["content_age"] = latest_release_year - netflix["release_year"]
netflix["is_recent_release"] = netflix["release_year"] >= (latest_release_year - 5)
netflix["is_classic_title"] = netflix["release_year"] < 2000
netflix["is_international"] = netflix["country_clean"].apply(lambda values: len([v for v in values if v != "Unknown Country"]) > 1)
netflix["genre_count"] = netflix["genre_clean"].apply(len)
netflix["country_count"] = netflix["country_clean"].apply(lambda values: len([v for v in values if v != "Unknown Country"]))
netflix["cast_count"] = netflix["cast"].apply(lambda value: 0 if value == "Unknown Cast" else len([item for item in str(value).split(",") if item.strip()]))
netflix["director_count"] = netflix["director"].apply(lambda value: 0 if value == "Unknown Director" else len([item for item in str(value).split(",") if item.strip()]))
netflix["has_director"] = netflix["director"].ne("Unknown Director")
netflix["has_cast"] = netflix["cast"].ne("Unknown Cast")
netflix["has_country"] = netflix["country"].ne("Unknown Country")

# Release era buckets support clean portfolio segmentation
netflix["release_era"] = pd.cut(
    netflix["release_year"],
    bins=[1900, 1979, 1989, 1999, 2009, 2015, latest_release_year],
    labels=["Pre-1980", "1980s", "1990s", "2000s", "2010-2015", "2016+"],
    include_lowest=True,
)

# Audience maturity segments based on common US-style Netflix ratings
maturity_map = {
    "G": "Kids/Family",
    "TV-Y": "Kids/Family",
    "TV-G": "Kids/Family",
    "PG": "Family/Teen",
    "TV-Y7": "Family/Teen",
    "TV-Y7-FV": "Family/Teen",
    "TV-PG": "Family/Teen",
    "PG-13": "Teen",
    "TV-14": "Teen",
    "R": "Mature",
    "NC-17": "Mature",
    "TV-MA": "Mature",
    "NR": "Unrated/Other",
    "UR": "Unrated/Other",
    "Unknown Rating": "Unrated/Other",
}
netflix["maturity_segment"] = netflix["rating"].map(maturity_map).fillna("Unrated/Other")

# Add timing flags for release planning analysis
netflix["added_quarter"] = netflix["date_added"].dt.quarter
netflix["added_season"] = netflix["month_added"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
})

feature_preview_cols = [
    "title", "type", "release_year", "content_age", "release_era",
    "maturity_segment", "genre_count", "country_count", "is_international",
    "cast_count", "director_count", "added_quarter", "added_season",
]
display(netflix[feature_preview_cols].head())

In [ ]:
engineered_summary = pd.DataFrame({
    "metric": [
        "Latest release year",
        "Recent release share",
        "Classic title share",
        "International/co-production share",
        "Average genres per title",
        "Average cast members listed",
        "Director metadata coverage",
        "Cast metadata coverage",
        "Country metadata coverage",
    ],
    "value": [
        latest_release_year,
        f"{netflix['is_recent_release'].mean() * 100:.1f}%",
        f"{netflix['is_classic_title'].mean() * 100:.1f}%",
        f"{netflix['is_international'].mean() * 100:.1f}%",
        f"{netflix['genre_count'].mean():.2f}",
        f"{netflix['cast_count'].mean():.2f}",
        f"{netflix['has_director'].mean() * 100:.1f}%",
        f"{netflix['has_cast'].mean() * 100:.1f}%",
        f"{netflix['has_country'].mean() * 100:.1f}%",
    ],
})

display(engineered_summary)

In [ ]:
display(Markdown(f'''
**Feature Engineering Insights**

- The dataset now includes **recency, catalog age, maturity, geography, genre breadth, and metadata completeness** features.
- **{netflix['is_recent_release'].mean() * 100:.1f}%** of titles are recent releases based on the latest release year in the dataset.
- **{netflix['is_international'].mean() * 100:.1f}%** of titles list multiple countries, indicating international collaboration or distribution metadata.

**Business implication:** These engineered features make it easier to evaluate catalog freshness, international reach, audience targeting, and metadata quality at portfolio level.
'''))

## Exploratory Data Analysis

### A. General Analysis

In [ ]:
total_titles = len(netflix)
type_counts = netflix["type"].value_counts()
type_percentages = (type_counts / total_titles * 100).round(2)

display(pd.DataFrame({"count": type_counts, "percentage": type_percentages}))
print(f"Total number of titles: {total_titles:,}")

In [ ]:
# Movies vs TV Shows count plot
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=netflix, x="type", order=type_counts.index, palette="Set2")
plt.title("Movies vs TV Shows on Netflix")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.savefig(IMAGE_DIR / "movie_vs_tvshow.png", bbox_inches="tight")
plt.show()

In [ ]:
# Percentage distribution pie chart
plt.figure(figsize=(7, 7))
plt.pie(
    type_counts,
    labels=type_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2", len(type_counts)),
)
plt.title("Percentage Distribution of Movies and TV Shows")
plt.tight_layout()
plt.show()

In [ ]:
dominant_type = type_counts.idxmax()
dominant_pct = type_percentages.loc[dominant_type]

display(Markdown(f'''
**General Analysis Insights**

- The catalog contains **{total_titles:,} total titles**.
- **{dominant_type}** are the dominant content type, representing **{dominant_pct:.2f}%** of the catalog.
- The balance between Movies and TV Shows indicates whether Netflix's catalog leans toward shorter one-time viewing or longer episodic engagement.

**Business implication:** A movie-heavy catalog can support variety and frequent browsing, while a stronger TV catalog can improve retention through multi-episode viewing.
'''))

### B. Time Analysis

In [ ]:
content_by_year = netflix.dropna(subset=["year_added"]).groupby("year_added").size()
content_by_month = netflix.dropna(subset=["month_added"]).groupby("month_added").size()
release_year_distribution = netflix.dropna(subset=["release_year"]).groupby("release_year").size()

display(content_by_year.tail(10).to_frame("titles_added"))

In [ ]:
# Content added per year and growth over time
plt.figure(figsize=(12, 6))
sns.lineplot(x=content_by_year.index, y=content_by_year.values, marker="o", color="#2F6B9A")
plt.title("Netflix Content Added per Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles Added")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "content_growth.png", bbox_inches="tight")
plt.show()

In [ ]:
# Content added per month
month_order = list(range(1, 13))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

plt.figure(figsize=(11, 5))
sns.barplot(x=month_order, y=[content_by_month.get(m, 0) for m in month_order], color="#72B7B2")
plt.xticks(ticks=range(12), labels=month_labels)
plt.title("Content Added by Month")
plt.xlabel("Month Added")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()

In [ ]:
# Release year distribution
plt.figure(figsize=(12, 5))
sns.histplot(data=netflix, x="release_year", bins=35, kde=True, color="#E45756")
plt.title("Release Year Distribution")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative content growth
cumulative_growth = content_by_year.cumsum()

plt.figure(figsize=(12, 6))
sns.lineplot(x=cumulative_growth.index, y=cumulative_growth.values, marker="o", color="#54A24B")
plt.title("Cumulative Netflix Catalog Growth")
plt.xlabel("Year Added")
plt.ylabel("Cumulative Titles")
plt.tight_layout()
plt.show()

In [ ]:
peak_year = int(content_by_year.idxmax())
peak_year_count = int(content_by_year.max())
peak_month = int(content_by_month.idxmax())
peak_month_name = pd.Timestamp(2024, peak_month, 1).month_name()

display(Markdown(f'''
**Time Analysis Insights**

- The highest content-addition year was **{peak_year}**, with **{peak_year_count:,} titles** added.
- The most active calendar month for additions was **{peak_month_name}**.
- Cumulative growth shows how aggressively Netflix expanded its catalog over time.

**Business implication:** Understanding addition patterns helps identify acquisition cycles, release calendar strategy, and periods of catalog acceleration or slowdown.
'''))

### C. Country Analysis

In [ ]:
# Explode country lists so multi-country titles are counted properly
country_exploded = netflix.explode("country_clean").rename(columns={"country_clean": "country_name"})
country_exploded = country_exploded[country_exploded["country_name"].ne("Unknown Country")]

top_countries = country_exploded["country_name"].value_counts().head(10)
display(top_countries.to_frame("title_count"))

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette="viridis")
plt.title("Top 10 Countries Producing Netflix Content")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "top_countries.png", bbox_inches="tight")
plt.show()

In [ ]:
country_type_counts = (
    country_exploded.groupby(["country_name", "type"])
    .size()
    .reset_index(name="count")
)

top_country_names = top_countries.index.tolist()
country_type_top = country_type_counts[country_type_counts["country_name"].isin(top_country_names)]

plt.figure(figsize=(12, 6))
sns.barplot(data=country_type_top, x="count", y="country_name", hue="type", palette="Set2")
plt.title("Country-wise Movie and TV Show Counts")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

In [ ]:
top_country = top_countries.index[0]
top_country_count = int(top_countries.iloc[0])

display(Markdown(f'''
**Country Analysis Insights**

- **{top_country}** contributes the largest number of titles, with **{top_country_count:,} appearances** in the country metadata.
- Comparing Movies and TV Shows by country highlights which markets are stronger in film versus episodic production.
- Multi-country titles are counted for each listed country, which better reflects international co-productions.

**Business implication:** Country-level patterns can guide regional licensing, localization, dubbing/subtitling priorities, and market-specific content investment.
'''))

### D. Genre Analysis

In [ ]:
# Explode genre lists for multi-label genre analysis
genre_exploded = netflix.explode("genre_clean").rename(columns={"genre_clean": "genre"})
top_genres = genre_exploded["genre"].value_counts().head(20)
display(top_genres.to_frame("title_count"))

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x=top_genres.values, y=top_genres.index, palette="mako")
plt.title("Top 20 Netflix Genres")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "top_genres.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=genre_exploded, y="genre", order=top_genres.head(10).index, hue="type", palette="Set2")
plt.title("Top 10 Genres by Content Type")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

In [ ]:
top_genre = top_genres.index[0]
top_genre_count = int(top_genres.iloc[0])

display(Markdown(f'''
**Genre Analysis Insights**

- **{top_genre}** is the most common genre, appearing in **{top_genre_count:,} titles**.
- Genre distribution shows which content categories dominate the Netflix catalog.
- Comparing genre mix by type reveals whether certain genres are more movie-oriented or TV-oriented.

**Business implication:** Genre demand and supply patterns can inform content acquisition, recommendation modules, and campaign targeting.
'''))

### E. Rating Analysis

In [ ]:
rating_counts = netflix["rating"].value_counts()
movie_ratings = netflix.query("type == 'Movie'")["rating"].value_counts()
tv_ratings = netflix.query("type == 'TV Show'")["rating"].value_counts()

display(rating_counts.to_frame("count"))

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=netflix, x="rating", order=rating_counts.index, palette="crest")
plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(IMAGE_DIR / "ratings_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=movie_ratings.head(10).index, y=movie_ratings.head(10).values, ax=axes[0], palette="Blues_r")
axes[0].set_title("Most Common Ratings for Movies")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(x=tv_ratings.head(10).index, y=tv_ratings.head(10).values, ax=axes[1], palette="Greens_r")
axes[1].set_title("Most Common Ratings for TV Shows")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
most_common_rating = rating_counts.index[0]
most_common_rating_count = int(rating_counts.iloc[0])

display(Markdown(f'''
**Rating Analysis Insights**

- **{most_common_rating}** is the most common rating, appearing in **{most_common_rating_count:,} titles**.
- Movie and TV rating profiles can differ, reflecting different audience targets and content maturity levels.
- Ratings distribution helps identify whether the catalog is concentrated around family, teen, or mature audiences.

**Business implication:** Ratings analysis supports parental-control design, personalized recommendations, and audience-specific marketing.
'''))

### F. Duration Analysis

In [ ]:
movies = netflix[(netflix["type"] == "Movie") & (netflix["duration_unit"].str.contains("min", na=False))]
tv_shows = netflix[(netflix["type"] == "TV Show") & (netflix["duration_unit"].str.contains("Season", na=False))]

display(movies["duration_value"].describe().to_frame("movie_duration_minutes"))
display(tv_shows["duration_value"].describe().to_frame("tv_show_seasons"))

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=movies, x="duration_value", bins=30, kde=True, color="#F58518")
plt.title("Movie Duration Distribution")
plt.xlabel("Duration in Minutes")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=movies, x="duration_value", color="#FFBF79")
plt.title("Movie Duration Box Plot")
plt.xlabel("Duration in Minutes")
plt.tight_layout()
plt.show()

In [ ]:
longest_movies = movies.sort_values("duration_value", ascending=False)[["title", "duration_value", "release_year", "rating"]].head(10)
display(longest_movies)

In [ ]:
season_distribution = tv_shows["duration_value"].value_counts().sort_index()

plt.figure(figsize=(11, 5))
sns.barplot(x=season_distribution.index, y=season_distribution.values, color="#59A14F")
plt.title("TV Show Seasons Distribution")
plt.xlabel("Number of Seasons")
plt.ylabel("Number of TV Shows")
plt.tight_layout()
plt.show()

In [ ]:
avg_movie_duration = movies["duration_value"].mean()
most_common_seasons = int(season_distribution.idxmax()) if not season_distribution.empty else 0

display(Markdown(f'''
**Duration Analysis Insights**

- The average movie duration is approximately **{avg_movie_duration:.1f} minutes**.
- The most common TV Show length is **{most_common_seasons} season(s)**.
- The longest movies can be reviewed separately because extremely long runtimes may represent special formats, documentaries, or incorrectly classified content.

**Business implication:** Runtime and season count influence viewing commitment, binge potential, and recommendation strategy.
'''))

### G. Director Analysis

In [ ]:
director_exploded = netflix.assign(
    director_clean=netflix["director"].str.split(",")
).explode("director_clean")
director_exploded["director_clean"] = director_exploded["director_clean"].str.strip()
director_exploded = director_exploded[director_exploded["director_clean"].ne("Unknown Director")]

top_directors = director_exploded["director_clean"].value_counts().head(15)
display(top_directors.to_frame("content_count"))

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x=top_directors.values, y=top_directors.index, palette="rocket")
plt.title("Top Directors by Netflix Content Count")
plt.xlabel("Number of Titles")
plt.ylabel("Director")
plt.tight_layout()
plt.show()

In [ ]:
if not top_directors.empty:
    top_director = top_directors.index[0]
    top_director_count = int(top_directors.iloc[0])
    director_text = f"**{top_director}** has the highest director count with **{top_director_count:,} titles**."
else:
    director_text = "Director metadata is not sufficient to identify top directors."

display(Markdown(f'''
**Director Analysis Insights**

- {director_text}
- Director frequency can reveal repeated partnerships or heavily licensed filmographies.

**Business implication:** Strong director patterns can support talent partnerships, creator-focused collections, and promotional campaigns.
'''))

### H. Cast Analysis

In [ ]:
cast_exploded = netflix.assign(
    cast_clean=netflix["cast"].str.split(",")
).explode("cast_clean")
cast_exploded["cast_clean"] = cast_exploded["cast_clean"].str.strip()
cast_exploded = cast_exploded[cast_exploded["cast_clean"].ne("Unknown Cast")]

top_actors = cast_exploded["cast_clean"].value_counts().head(20)
display(top_actors.to_frame("appearance_count"))

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x=top_actors.values, y=top_actors.index, palette="cubehelix")
plt.title("Most Frequent Actors Appearing on Netflix")
plt.xlabel("Number of Appearances")
plt.ylabel("Actor")
plt.tight_layout()
plt.show()

In [ ]:
if not top_actors.empty:
    top_actor = top_actors.index[0]
    top_actor_count = int(top_actors.iloc[0])
    actor_text = f"**{top_actor}** appears most frequently, with **{top_actor_count:,} titles**."
else:
    actor_text = "Cast metadata is not sufficient to identify top actors."

display(Markdown(f'''
**Cast Analysis Insights**

- {actor_text}
- High-frequency cast members may reflect strong regional catalogs, long-running franchises, or popular licensing groups.

**Business implication:** Actor-level analysis can support star-led recommendation rows, regional campaigns, and audience acquisition.
'''))

### I. Content Type Trends

In [ ]:
type_year = (
    netflix.dropna(subset=["year_added"])
    .groupby(["year_added", "type"])
    .size()
    .reset_index(name="count")
)

type_year_pivot = type_year.pivot(index="year_added", columns="type", values="count").fillna(0)
display(type_year_pivot.tail(10))

In [ ]:
plt.figure(figsize=(12, 6))
for content_type in type_year_pivot.columns:
    sns.lineplot(x=type_year_pivot.index, y=type_year_pivot[content_type], marker="o", label=content_type)

plt.title("Movies vs TV Shows Added Over Years")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles Added")
plt.legend(title="Type")
plt.tight_layout()
plt.show()

In [ ]:
type_year_pivot.plot(kind="bar", stacked=True, figsize=(13, 6), colormap="Set2")
plt.title("Stacked Bar Chart: Movies vs TV Shows Over Years")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles Added")
plt.legend(title="Type")
plt.tight_layout()
plt.show()

In [ ]:
latest_year = int(type_year_pivot.index.max())
latest_mix = type_year_pivot.loc[latest_year].sort_values(ascending=False)
latest_top_type = latest_mix.index[0]

display(Markdown(f'''
**Content Type Trend Insights**

- In the latest available addition year (**{latest_year}**), **{latest_top_type}** had the highest count among content types.
- Trend lines show whether Netflix's catalog expansion was more movie-led or TV-led in different periods.
- Stacked bars make yearly portfolio mix shifts easier to compare.

**Business implication:** Content type trends help guide investment between broad movie variety and longer-form episodic engagement.
'''))

### J. Correlation and Advanced Analysis

In [ ]:
# Numeric correlation heatmap where applicable
numeric_cols = ["release_year", "year_added", "month_added", "duration_value", "content_age_when_added"]
numeric_data = netflix[numeric_cols].copy()
correlation_matrix = numeric_data.corr()

plt.figure(figsize=(9, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
# Cross-tab analysis: content type vs rating
type_rating_crosstab = pd.crosstab(netflix["rating"], netflix["type"])
display(type_rating_crosstab.sort_values(by=type_rating_crosstab.columns.tolist(), ascending=False).head(15))

plt.figure(figsize=(9, 7))
sns.heatmap(type_rating_crosstab, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Content Type vs Rating Heatmap")
plt.xlabel("Content Type")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Groupby summaries
groupby_summary = (
    netflix.groupby("type")
    .agg(
        title_count=("show_id", "count"),
        avg_release_year=("release_year", "mean"),
        avg_year_added=("year_added", "mean"),
        avg_duration_value=("duration_value", "mean"),
        median_duration_value=("duration_value", "median"),
    )
    .round(2)
)

display(groupby_summary)

In [ ]:
# Genre and rating cross-tab for top genres
top_10_genres = top_genres.head(10).index
genre_rating = pd.crosstab(
    genre_exploded[genre_exploded["genre"].isin(top_10_genres)]["genre"],
    genre_exploded[genre_exploded["genre"].isin(top_10_genres)]["rating"],
)

plt.figure(figsize=(14, 7))
sns.heatmap(genre_rating, cmap="Blues", linewidths=0.3)
plt.title("Top Genres vs Ratings Heatmap")
plt.xlabel("Rating")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown('''
**Advanced Analysis Insights**

- Correlation analysis helps identify relationships between release year, addition year, and duration-related features.
- Rating cross-tabs show how audience suitability differs across Movies and TV Shows.
- Groupby summaries provide a compact comparison of content types across time and duration metrics.

**Business implication:** Advanced summaries help move beyond single-variable charts toward portfolio segmentation and decision-ready analysis.
'''))

### K. Statistical Observations

In [ ]:
# Statistical observations for Movies and TV Shows
movie_duration_stats = movies["duration_value"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2)
movie_iqr = movie_duration_stats.loc["75%"] - movie_duration_stats.loc["25%"]
long_movie_threshold = movie_duration_stats.loc["75%"] + 1.5 * movie_iqr
long_movie_outliers = movies[movies["duration_value"] > long_movie_threshold]

tv_season_stats = tv_shows["duration_value"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2)
content_age_stats = netflix["content_age_when_added"].dropna().describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2)

statistical_summary = pd.DataFrame({
    "metric": [
        "Movie median duration",
        "Movie IQR duration",
        "Long movie outlier threshold",
        "Long movie outlier count",
        "TV Show median seasons",
        "Median content age when added",
        "90th percentile content age when added",
    ],
    "value": [
        f"{movie_duration_stats.loc['50%']:.1f} min",
        f"{movie_iqr:.1f} min",
        f"{long_movie_threshold:.1f} min",
        f"{len(long_movie_outliers):,}",
        f"{tv_season_stats.loc['50%']:.1f} season(s)" if not tv_season_stats.empty else "N/A",
        f"{content_age_stats.loc['50%']:.1f} years",
        f"{content_age_stats.loc['90%']:.1f} years",
    ],
})

display(statistical_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=movies, x="rating", y="duration_value", ax=axes[0], palette="Set3")
axes[0].set_title("Movie Runtime by Rating", fontweight="bold")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Duration in Minutes")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=netflix.dropna(subset=["content_age_when_added"]), x="type", y="content_age_when_added", ax=axes[1], palette="Set2")
axes[1].set_title("Content Age When Added by Type", fontweight="bold")
axes[1].set_xlabel("Content Type")
axes[1].set_ylabel("Years Between Release and Netflix Addition")

plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(f'''
**Statistical Observations**

- The median movie runtime is **{movie_duration_stats.loc['50%']:.1f} minutes**, with an interquartile range of **{movie_iqr:.1f} minutes**.
- Movies above **{long_movie_threshold:.1f} minutes** are statistical high-runtime outliers by the 1.5x IQR rule.
- The median title was added about **{content_age_stats.loc['50%']:.1f} years** after its release.

**Business implication:** Runtime and content-age outliers can be reviewed separately for special programming, niche audiences, licensing value, or metadata quality checks.
'''))

### L. Advanced Trend Analysis

In [ ]:
yearly_trends = (
    netflix.dropna(subset=["year_added"])
    .groupby("year_added")
    .agg(
        titles_added=("show_id", "count"),
        movie_share=("type", lambda values: (values == "Movie").mean() * 100),
        tv_share=("type", lambda values: (values == "TV Show").mean() * 100),
        avg_content_age=("content_age_when_added", "mean"),
        mature_share=("maturity_segment", lambda values: (values == "Mature").mean() * 100),
        international_share=("is_international", "mean"),
    )
    .assign(
        yoy_growth_pct=lambda x: x["titles_added"].pct_change() * 100,
        international_share=lambda x: x["international_share"] * 100,
    )
    .round(2)
)

display(yearly_trends.tail(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.lineplot(data=yearly_trends, x=yearly_trends.index, y="titles_added", marker="o", ax=axes[0, 0], color=BRAND_COLORS["red"])
axes[0, 0].set_title("Annual Content Additions", fontweight="bold")
axes[0, 0].set_xlabel("Year Added")
axes[0, 0].set_ylabel("Titles Added")

sns.lineplot(data=yearly_trends, x=yearly_trends.index, y="yoy_growth_pct", marker="o", ax=axes[0, 1], color=BRAND_COLORS["blue"])
axes[0, 1].axhline(0, color=BRAND_COLORS["black"], linewidth=1, linestyle="--")
axes[0, 1].set_title("Year-over-Year Growth Rate", fontweight="bold")
axes[0, 1].set_xlabel("Year Added")
axes[0, 1].set_ylabel("YoY Growth (%)")

sns.lineplot(data=yearly_trends, x=yearly_trends.index, y="movie_share", marker="o", ax=axes[1, 0], label="Movies", color=BRAND_COLORS["orange"])
sns.lineplot(data=yearly_trends, x=yearly_trends.index, y="tv_share", marker="o", ax=axes[1, 0], label="TV Shows", color=BRAND_COLORS["green"])
axes[1, 0].set_title("Content Mix Share Over Time", fontweight="bold")
axes[1, 0].set_xlabel("Year Added")
axes[1, 0].set_ylabel("Share of Annual Additions (%)")

sns.lineplot(data=yearly_trends, x=yearly_trends.index, y="avg_content_age", marker="o", ax=axes[1, 1], color=BRAND_COLORS["gray"])
axes[1, 1].set_title("Average Content Age When Added", fontweight="bold")
axes[1, 1].set_xlabel("Year Added")
axes[1, 1].set_ylabel("Average Years Since Release")

for ax in axes.flat:
    ax.grid(axis="y", alpha=0.25)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
strongest_growth_year = int(yearly_trends["yoy_growth_pct"].idxmax())
strongest_growth_value = yearly_trends.loc[strongest_growth_year, "yoy_growth_pct"]
latest_yoy = yearly_trends["yoy_growth_pct"].dropna().iloc[-1]

display(Markdown(f'''
**Trend Analysis Insights**

- The strongest year-over-year growth occurred in **{strongest_growth_year}**, when annual additions increased by **{strongest_growth_value:.1f}%**.
- The latest available year shows **{latest_yoy:.1f}%** year-over-year change in title additions.
- Tracking content age alongside volume helps distinguish fresh-release acquisition from library expansion.

**Business implication:** Trend analysis helps leadership identify expansion periods, slowdown signals, and whether catalog growth is driven by new releases or older licensed inventory.
'''))

### M. Portfolio Segmentation

In [ ]:
maturity_type = pd.crosstab(netflix["maturity_segment"], netflix["type"], normalize="columns") * 100
era_type = pd.crosstab(netflix["release_era"], netflix["type"])
quarter_type = pd.crosstab(netflix["added_quarter"], netflix["type"])

display(maturity_type.round(2))
display(era_type)
display(quarter_type)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(maturity_type.round(1), annot=True, fmt=".1f", cmap="Reds", ax=axes[0])
axes[0].set_title("Maturity Segment Share by Type (%)", fontweight="bold")
axes[0].set_xlabel("Content Type")
axes[0].set_ylabel("Maturity Segment")

sns.heatmap(era_type, annot=True, fmt="d", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Release Era by Content Type", fontweight="bold")
axes[1].set_xlabel("Content Type")
axes[1].set_ylabel("Release Era")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
quarter_type.plot(kind="bar", stacked=True, colormap="Set2", ax=plt.gca())
polish_chart("Quarterly Release Cadence by Content Type", "Quarter Added", "Number of Titles")
plt.legend(title="Type")
plt.show()

In [ ]:
leading_maturity = netflix["maturity_segment"].value_counts().idxmax()
leading_era = netflix["release_era"].value_counts().idxmax()

display(Markdown(f'''
**Portfolio Segmentation Insights**

- **{leading_maturity}** is the largest audience maturity segment in the catalog.
- **{leading_era}** is the most common release-era bucket.
- Quarterly patterns reveal whether content additions are evenly distributed or concentrated in specific planning windows.

**Business implication:** Portfolio segmentation supports more precise content rows, audience targeting, compliance review, and quarterly campaign planning.
'''))

### N. Genre and Country Opportunity Matrix

In [ ]:
top_12_genres = top_genres.head(12).index
top_8_countries = top_countries.head(8).index

genre_country_matrix = (
    genre_exploded.explode("country_clean")
    .rename(columns={"country_clean": "country_name"})
    .query("genre in @top_12_genres and country_name in @top_8_countries")
    .pivot_table(index="genre", columns="country_name", values="show_id", aggfunc="count", fill_value=0)
)

plt.figure(figsize=(14, 8))
sns.heatmap(genre_country_matrix, annot=True, fmt="d", cmap="rocket_r", linewidths=0.4)
plt.title("Genre and Country Opportunity Matrix", fontsize=16, fontweight="bold", pad=14)
plt.xlabel("Country")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

In [ ]:
genre_type_share = pd.crosstab(genre_exploded["genre"], genre_exploded["type"], normalize="index") * 100
genre_type_share = genre_type_share.loc[top_12_genres].round(1)

plt.figure(figsize=(12, 7))
genre_type_share.sort_values(by="Movie", ascending=True).plot(kind="barh", stacked=True, colormap="Set2", ax=plt.gca())
polish_chart("Movie vs TV Show Share Within Top Genres", "Share Within Genre (%)", "Genre")
plt.legend(title="Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
display(Markdown('''
**Genre and Country Opportunity Insights**

- The genre-country heatmap highlights where specific content categories are most concentrated.
- Genre type share shows which genres are mainly movie-led versus TV-led.
- Underrepresented genre-country combinations may represent acquisition gaps or deliberate strategic choices.

**Business implication:** This matrix can guide region-specific genre investments, licensing negotiations, and localized merchandising campaigns.
'''))

### O. Metadata Quality Analysis

In [ ]:
metadata_quality = pd.DataFrame({
    "field": ["director", "cast", "country", "rating", "date_added"],
    "coverage_pct": [
        netflix["has_director"].mean() * 100,
        netflix["has_cast"].mean() * 100,
        netflix["has_country"].mean() * 100,
        netflix["rating"].ne("Unknown Rating").mean() * 100,
        netflix["date_added"].notna().mean() * 100,
    ],
}).sort_values("coverage_pct")

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=metadata_quality, x="coverage_pct", y="field", palette="crest")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
polish_chart("Metadata Coverage by Field", "Coverage (%)", "Field")
plt.xlim(0, 105)
plt.show()

display(metadata_quality.round(2))

In [ ]:
weakest_metadata_field = metadata_quality.iloc[0]["field"]
weakest_metadata_pct = metadata_quality.iloc[0]["coverage_pct"]

display(Markdown(f'''
**Metadata Quality Insights**

- **{weakest_metadata_field}** has the lowest metadata coverage at **{weakest_metadata_pct:.1f}%**.
- Missing metadata can weaken search, recommendations, attribution, and regional reporting.

**Business implication:** Improving low-coverage metadata fields should be treated as a data-quality priority because it directly affects product discovery and analytics accuracy.
'''))

## Executive Summary

In [ ]:
executive_summary = f'''
### Executive Summary

Netflix's catalog in this dataset contains **{total_titles:,} titles**, with **{dominant_type}** making up the largest share at **{dominant_pct:.2f}%**. Catalog growth peaked in **{peak_year}**, when **{peak_year_count:,} titles** were added, and the strongest year-over-year growth was observed in **{strongest_growth_year}**.

From a portfolio perspective, **{top_country}** is the leading content-producing country, **{top_genre}** is the most common genre, and **{most_common_rating}** is the most frequent rating. The catalog's median movie runtime is **{movie_duration_stats.loc['50%']:.1f} minutes**, and the median content age when added is **{content_age_stats.loc['50%']:.1f} years**.

The analysis suggests three practical priorities:

1. Balance high-volume genres with underrepresented opportunity areas by country.
2. Use maturity and duration segments to strengthen audience-specific recommendations.
3. Improve metadata completeness, especially for the weakest field: **{weakest_metadata_field}**.
'''

display(Markdown(executive_summary))

## Insights and Conclusions

In [ ]:
# Build final findings dynamically from the cleaned dataset
findings = []

findings.append(f"The Netflix catalog contains {total_titles:,} titles in this dataset.")
findings.append(f"{dominant_type} are the largest content type, representing {dominant_pct:.2f}% of the catalog.")
findings.append(f"The peak year for content additions was {peak_year}, with {peak_year_count:,} titles added.")
findings.append(f"{peak_month_name} was the most active month for adding content.")
findings.append(f"{top_country} is the leading listed country, appearing in {top_country_count:,} titles.")
findings.append(f"{top_genre} is the most frequent genre, appearing in {top_genre_count:,} titles.")
findings.append(f"{most_common_rating} is the most common content rating, with {most_common_rating_count:,} titles.")
findings.append(f"The average movie duration is approximately {avg_movie_duration:.1f} minutes.")
findings.append(f"The most common TV Show length is {most_common_seasons} season(s).")
findings.append(f"In {latest_year}, {latest_top_type} had the highest additions among content types.")

recommendations = [
    "Strengthen investment in high-performing genres while maintaining enough variety for discovery.",
    "Use country-level patterns to prioritize regional licensing, localization, and original production.",
    "Monitor the balance between Movies and TV Shows to support both browsing variety and subscriber retention.",
    "Use ratings distribution to improve family, teen, and mature-audience recommendation experiences.",
    "Promote frequent actors and directors through curated collections and talent-led campaigns.",
    "Investigate years with sharp catalog growth or decline to understand acquisition strategy changes.",
    "Use duration and season patterns to create viewing-time based recommendation rows.",
    "Improve metadata completeness for cast, director, and country fields to strengthen search and personalization.",
]

markdown_text = "### 10 Key Findings\n\n"
markdown_text += "\n".join([f"{i+1}. {finding}" for i, finding in enumerate(findings)])
markdown_text += "\n\n### Actionable Recommendations\n\n"
markdown_text += "\n".join([f"{i+1}. {rec}" for i, rec in enumerate(recommendations)])
markdown_text += (
    "\n\n### Summary of Trends\n\n"
    "Netflix's catalog can be understood through four major lenses: content type balance, "
    "time-based growth, geographic contribution, and genre/rating mix. Together, these "
    "trends show how Netflix has built a broad international catalog while balancing "
    "audience maturity levels, regional supply, and content formats."
)

display(Markdown(markdown_text))